# Practical Exam: Automating Customer Support with OpenAI API

You work as an AI Engineer at ChatSolveAI, a company that provides automated customer support solutions. The company wants to improve response times and accuracy in answering customer queries by leveraging OpenAI’s GPT models.

Your task is to build a chatbot that classifies customer queries, retrieves relevant responses, and logs interactions in a structured way. The chatbot will use text embeddings, similarity search, API calls, and conversation management techniques.


**Please note:** 

1. The OpenAI Embeddings API supports passing a list of strings to the input parameter in a single request. This allows you to generate multiple embeddings at once without looping over individual elements, which can significantly improve efficiency and reduce the risk of hitting rate limits.

2. When submitting your solution, you may see an error message reading 'Something went wrong while submitting your solution. Please try again.' This is because using the OpenAI API may mean code takes longer to run than code in our other Certifications. Please ignore this message if your code is taking a few minutes to run. However, if your code makes too many API requests, the API will time out. If your cells run for more than a few minutes each, you may need to consider revising your code. 

In [8]:
# Run this cell before running your solution

# Import necessary modules
import os
from openai import OpenAI

# Define the model to use
model = "gpt-3.5-turbo"

# Define the client
client = OpenAI()

# Task 1

ChatSolveAI has provided a knowledge base (`knowledge_base.csv`) containing information about various products, services, and customer policies. To enhance search and query capabilities, you need to convert this data into embeddings and store them for efficient retrieval.

- Load the dataset (`knowledge_base.csv`).
- Generate text embeddings using OpenAI’s embedding model (`text-embedding-3-small`). Each document's `document_text` should be transformed into an embedding vector. Do not apply any text transformations such as lowercasing, stripping or normalization before embedding.
- Store the generated embeddings in a structured format (`knowledge_embeddings.json`) with the following format available below.
- Store the embedded data and associated metadata for retrieval.  

### Format to store generated embeddings:
```json
[
    {
       "document_id": 1,
       "document_text": "Example document text.",
       "embedding_vector": [0.123, 0.456, ...],
       "metadata": "Additional document info"
    }
]
```

### Data description: 

| Column Name       | Criteria                                                |
|-------------------|---------------------------------------------------------|
| document_id       | Integer. Unique identifier for each document. No missing values. |
| document_text     | String. Text content of the knowledge base. Preprocessed and embedded. |
| embedding_vector  | List. Embedding representation of the `document_text`. |
| metadata          | String. Metadata for additional information. |


In [9]:
# Write your answer to Task 1 here

import json
import pandas as pd

#Step 1: This is were we define the model embeddings 
embedding_model = "text-embedding-3-small"

#Step 2: This is were we load the datasets
kb_df = pd.read_csv("knowledge_base.csv")

#Step 3: This is were we generate the embeddings 
document_texts = kb_df["document_text"].tolist()

response = client.embeddings.create(
    model=embedding_model,
    input=document_texts
)

#Step 4: Structuring the outputs 
knowledge_embeddings = []
for idx, row in kb_df.iterrows():
    knowledge_embeddings.append({
        "document_id": int(row["document_id"]),
        "document_text": row["document_text"],
        "embedding_vector": response.data[idx].embedding,
        "metadata": row["metadata"]
    })

#Step 5: Save to Json file 
with open("knowledge_embeddings.json", "w") as f:
    json.dump(knowledge_embeddings, f)

# Task 2

ChatSolveAI receives customer queries that need to be classified and matched with appropriate responses. Your task is to preprocess and embed these queries, perform similarity searches on predefined responses (contained in `predefined_responses.json`), and retrieve the most relevant responses.

- Load the dataset (`processed_queries.csv`).
- Retrieve responses by using cosine similarity to perform a similarity search against predefined responses in `predefined_responses.json`.
- Structure API requests properly and implement error handling, including retry mechanisms to handle rate limits.
- Format model responses as JSON to maintain consistency in output.
- Compute confidence scores for retrieved responses, scaled to 0-1.
- Store the structured responses in a JSON file (`query_responses.json`), suitable for integration with other applications. Your JSON file should be structured as follows:

| Column Name       | Criteria                                                   |
|-------------------|------------------------------------------------------------|
| query_id         | Integer. Unique identifier for each query. No missing values. |
| query_text       | String. Preprocessed query text. |
| top_responses    | List. Top 3 most relevant responses retrieved. |
| confidence_scores | List. Model-based confidence score for the top 3 responses. |

In [10]:
# Write your answer to Task 2 here

import json
import time
import pandas as pd
import numpy as np

#Step 1: Defining the embeddings model 
embedding_model = "text-embedding-3-small"

#Step 2: cosine 
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot_product / (norm1 * norm2)

#Step 3: Getting embeddings with retry 
def get_embeddings_with_retry(texts, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.embeddings.create(
                model=embedding_model,
                input=texts
            )
            return [data.embedding for data in response.data]
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                raise e

#Step 4: Load datasets 
queries_df = pd.read_csv("processed_queries.csv")

with open("predefined_responses.json", "r") as f:
    predefined_responses = json.load(f)

response_texts = list(predefined_responses.values())

#Step 5: Batch processing 
query_texts = queries_df["query_text"].tolist()
query_embeddings = get_embeddings_with_retry(query_texts)
response_embeddings = get_embeddings_with_retry(response_texts)

#Step 6: Similarity Search 
query_responses = []

for i, query_emb in enumerate(query_embeddings):
    similarities = []
    for j, resp_emb in enumerate(response_embeddings):
        sim = cosine_similarity(query_emb, resp_emb)
        similarities.append((j, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    top_3 = similarities[:3]
    
    top_responses = [response_texts[idx] for idx, _ in top_3]
    confidence_scores = [float(score) for _, score in top_3]
    
    query_responses.append({
        "query_id": int(queries_df.iloc[i]["query_id"]),
        "query_text": queries_df.iloc[i]["query_text"],
        "top_responses": top_responses,
        "confidence_scores": confidence_scores
    })

#Step 7: Load to Json file 
with open("query_responses.json", "w") as f:
    json.dump(query_responses, f, indent=4)

# Task 3

To provide seamless customer service, ChatSolveAI wants to develop a chatbot that can respond to customer queries efficiently by searching for relevant responses and generating new ones when necessary.

- Develop a chatbot that:
    - Accepts customer queries via text input.
    - Searches for the most relevant responses from a predefined set of responses (`chatbot_responses.json`).
    - Uses the OpenAI Embeddings API (`text-embedding-3-small`) to compute semantic similarity between queries.
    - If no relevant response is found from the predefined set, generates a new response using GPT-3.5-turbo.
- Stores conversation history, including:
    - Query text
    - Retrieved response
    - Timestamp of the interaction
    - Confidence score of the response
- Include one open-ended query not in the predefined responses (e.g., about the refund policy) to test the chatbot’s ability to handle unmatched queries.
- Include one paraphrased query about support hours (e.g., “When can I talk to someone from support?”) to test semantic similarity matching.
- Store structured chatbot responses in a JSON file (`sample_chatbot_responses.json`). Make sure they follow this format:
```json
[
    {
        "query_text": "How do I reset my password?",
        "retrieved_response": "You can reset your password by clicking 'Forgot Password' on the login page.",
        "timestamp": "2025-04-02T14:30:00Z",
        "confidence_score": 0.92
    },
    {
        "query_text": "What are your business hours?",
        "retrieved_response": "Our support team is available from 9 AM to 5 PM, Monday to Friday.",
        "timestamp": "2025-04-02T14:35:00Z",
        "confidence_score": 0.87
    }
]
```

In [11]:
# Write your answer to Task 3 here

import json
import numpy as np
from datetime import datetime, timezone

embedding_model = "text-embedding-3-small"

def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot_product / (norm1 * norm2)

#Step 1: Load the chatbot knowledge base 
with open("chatbot_responses.json", "r") as f:
    chatbot_kb = json.load(f)

#Step 2: Generate KB queries 
kb_query_texts = [item["query_text"] for item in chatbot_kb]
kb_emb_response = client.embeddings.create(
    model=embedding_model,
    input=kb_query_texts
)
kb_embeddings = [data.embedding for data in kb_emb_response.data]

#Step 3: Chatbot functions 
def get_chatbot_response(user_query, threshold=0.75):
    user_emb_response = client.embeddings.create(
        model=embedding_model,
        input=[user_query]
    )
    user_embedding = user_emb_response.data[0].embedding
    
    best_score = -1
    best_idx = 0
    for i, kb_emb in enumerate(kb_embeddings):
        sim = cosine_similarity(user_embedding, kb_emb)
        if sim > best_score:
            best_score = sim
            best_idx = i
    
    timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    
    if best_score >= threshold:
        retrieved_response = chatbot_kb[best_idx]["retrieved_response"]
    else:
        completion = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a helpful customer support assistant. Answer clearly and concisely."},
                {"role": "user", "content": user_query}
            ],
            max_tokens=150,
            temperature=0.7
        )
        retrieved_response = completion.choices[0].message.content
    
    return {
        "query_text": user_query,
        "retrieved_response": retrieved_response,
        "timestamp": timestamp,
        "confidence_score": float(round(best_score, 2))
    }

#Step 4: Defining the chatbot queries 
test_queries = [
    "How do I reset my password?",
    "What are your business hours?",
    "When can I talk to someone from support?",
    "Can I get a refund if I downloaded the software?"
]

#Step 5: Processing and storing 
conversation_history = []
for query in test_queries:
    result = get_chatbot_response(query)
    conversation_history.append(result)

#Step 6: Dont forget to save 
with open("sample_chatbot_responses.json", "w") as f:
    json.dump(conversation_history, f, indent=4)